In [1]:
import os
from datetime import datetime
import pickle
import random
import math
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.colors as mcolors
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler, MaxAbsScaler, MinMaxScaler

import torch
import torch.nn as nn
import torch.nn.functional as F


In [2]:
df = pd.read_csv(f'BTCUSDT-15m-data.csv')
print(df.head())

             timestamp      open_time  ...   tb_quote_av  ignore
0  2017-08-17 04:00:00  1502942400000  ...   2089.104962       0
1  2017-08-17 04:15:00  1502943300000  ...  14703.934995       0
2  2017-08-17 04:30:00  1502944200000  ...  87620.977876       0
3  2017-08-17 04:45:00  1502945100000  ...  46538.460109       0
4  2017-08-17 05:00:00  1502946000000  ...  15093.783057       0

[5 rows x 13 columns]


In [3]:
df.describe()

,open_time,open,high,low,close,volume,close_time,quote_av,trades,tb_base_av,tb_quote_av,ignore
count,2.760000e+05,276000.000000,276000.000000,276000.000000,276000.000000,276000.000000,2.760000e+05,2.760000e+05,276000.000000,276000.000000,2.760000e+05,276000.0
mean,1.627546e+12,31546.615802,31614.327921,31476.715658,31546.982447,680.883620,1.627547e+12,1.831719e+07,18357.087167,338.333706,9.058558e+06,0.0
std,7.182939e+10,27430.002253,27476.745667,27382.290597,27430.355126,1090.897438,7.182939e+10,2.874946e+07,27465.292542,544.280131,1.442830e+07,0.0
min,1.502942e+12,2830.000000,2880.010000,2817.000000,2820.000000,0.000000,1.502943e+12,0.000000e+00,0.000000,0.000000,0.000000e+00,0.0
25%,1.565369e+12,8830.617500,8854.872500,8805.007500,8830.710000,187.694860,1.565370e+12,2.897420e+06,3490.750000,91.801213,1.430186e+06,0.0
50%,1.627623e+12,22939.860000,22984.725000,22887.405000,22939.860000,350.540700,1.627623e+12,8.657732e+06,8852.000000,174.819142,4.136309e+06,0.0
75%,1.689750e+12,47075.607500,47197.020000,46937.152500,47076.182500,705.999195,1.689751e+12,2.244964e+07,20720.250000,351.470489,1.106094e+07,0.0
max,1.751850e+12,111898.740000,111980.000000,111681.810000,111898.740000,40371.405060,1.751851e+12,8.961201e+08,606234.000000,19925.616600,5.619115e+08,0.0


In [4]:
df.tail()

,timestamp,open_time,open,high,low,close,volume,close_time,quote_av,trades,tb_base_av,tb_quote_av,ignore
275995,2025-07-07 00:00:00,1751846400000,109203.85,109273.07,109103.19,109273.06,56.48447,1751847299999,6.166301e+06,26415,26.22332,2.862917e+06,0
275996,2025-07-07 00:15:00,1751847300000,109273.07,109288.02,108972.69,109004.81,75.36437,1751848199999,8.220317e+06,24191,33.56157,3.660548e+06,0
275997,2025-07-07 00:30:00,1751848200000,109004.81,109037.82,108829.17,108847.16,76.72080,1751849099999,8.357724e+06,21368,33.08997,3.605029e+06,0
275998,2025-07-07 00:45:00,1751849100000,108847.17,108922.00,108800.01,108823.07,45.06548,1751849999999,4.906468e+06,15579,22.76836,2.478881e+06,0
275999,2025-07-07 01:00:00,1751850000000,108823.07,108823.08,108679.75,108767.01,75.71845,1751850899999,8.232864e+06,22928,38.85713,4.224608e+06,0


In [5]:
print(df.isnull())
print(f'Counts how many missing values there are in each column: {df.isnull().sum()}')
print(f'Total missing values: {df.isnull().sum().sum()}')

        timestamp  open_time   open  ...  tb_base_av  tb_quote_av  ignore
0           False      False  False  ...       False        False   False
1           False      False  False  ...       False        False   False
2           False      False  False  ...       False        False   False
3           False      False  False  ...       False        False   False
4           False      False  False  ...       False        False   False
...           ...        ...    ...  ...         ...          ...     ...
275995      False      False  False  ...       False        False   False
275996      False      False  False  ...       False        False   False
275997      False      False  False  ...       False        False   False
275998      False      False  False  ...       False        False   False
275999      False      False  False  ...       False        False   False

[276000 rows x 13 columns]
Counts how many missing values there are in each column: timestamp      0
open_time 

#Training Phase

Splitting data to train/test sets

Practice same pattersn multiple times

Have access to answer key

problems known:
Overfitting
Underfitting

#Testing Phase

Face new Patterns

No access to asnwer key


In [6]:
train_end_idx = 240_000

df_train = df.iloc[:train_end_idx].copy()

df_test = df.iloc[train_end_idx:].copy()

df_test.reset_index(drop=True, inplace=True)

# Instead o preprocessing the data set first
# we split it and then preprocess
# this way we can ensure the test data do not leak into the training data

print(f'Length of training set: {len(df_train)}')
print(f'Length of testing set: {len(df_test)}')

Length of training set: 240000
Length of testing set: 36000


In [7]:
# what is a feature:
# a Mesurable piece of information that describe the data
# help the model to recognize patterns

#always pay attention to the shape of the data

def build_feature(opens, closes):
  fearure1 = (closes - opens) / opens

  features = np.stack([
      fearure1,
  ], axis=-1) # shape: (time_steps, num_features)

  num_features = features.shape[-1] # shape -1: means last dimentsion of the features array that correxpond to the n of features

  return features, num_features

In [10]:
def preprocessing_data(seq_len, df):
  m =len(df)

  opens = np.array(df['open'].values)
  closes = np.array(df['close'].values)

  features, num_features = build_feature(opens, closes)



  # calculate number of samples
  num_samples = m - seq_len

  #create storage for input & targets
  # x the input sequence
  # y the output/target
  X = np.zeros([num_samples, seq_len, num_features], dtype=np.float32)
  Y = np.zeros([num_samples, num_features], dtype=np.float32)

  for i in range(num_samples):
    X[i] = features[i:i+seq_len]
    Y[i] = features[i+seq_len : i+seq_len+1]

  return X, Y, num_features

In [11]:
from re import X
# Sequence Lenght fot the inout sequence equivalent 1 day of 15m candles
seq_len = 96

# Processing data
X_train, Y_train, num_features = preprocessing_data(seq_len, df_train)
X_test, Y_test, num_features = preprocessing_data(seq_len, df_test)

m_train = X_train.shape[0]
m_test = X_test.shape[0]


print(f'Number of training samples (m_train): {m_train}')
print(f'Number of testing samples (m_test): {m_test}')
print(f'X_train shape: {X_train.shape}')
print(f'Y_train shape: {Y_train.shape}')
print(f'X_test shape: {X_test.shape}')
print(f'Y_test shape: {Y_test.shape}')

Number of training samples (m_train): 239904
Number of testing samples (m_test): 35904
X_train shape: (239904, 96, 1)
Y_train shape: (239904, 1)
X_test shape: (35904, 96, 1)
Y_test shape: (35904, 1)


In [12]:
# pytorch models can only work with tensors

# pytorch and tensorFlow

#what is a tensor: a similar to a numpy array, but it can be on a cpu

# pytorch works best with float 32

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print("my device: ", device)

X_train = torch.from_numpy(X_train.astype(np.float32)).to(device, dtype=torch.float32)
Y_train = torch.from_numpy(Y_train.astype(np.float32)).to(device, dtype=torch.float32)

X_test = torch.from_numpy(X_test.astype(np.float32)).to(device, dtype=torch.float32)
Y_test = torch.from_numpy(Y_test.astype(np.float32)).to(device, dtype=torch.float32)

Y_pred_test = torch.zeros([m_test, num_features], device=device, dtype=torch.float32)


my device:  cpu
